# 16 — BCG mouse data preprocessing

Re-align Josh's BCG mouse object to the **atlas-full HVG gene namespace** (per flavor) for prediction. This notebook maps BCG **Ensembl gene ids** (`var['gene_ids']`) → human **ENSG** via the cached atlas ortholog table, subsets to the same 1k HVG list as notebook **15**, and writes CellOT-ready `h5ad` files.

---

## Design decision: keep mentor `.X` (no count swap, no `log1p`)

The source object (`bcg_mouse_aligned_050626.h5ad`) is produced in **`tb1_mouse_setup.ipynb`**: after **scVI/scANVI**, **`.X`** is **`get_normalized_expression(..., transform_batch="atlas", library_size=1e4)`** (and optional per-gene anchor scaling), i.e. **expression already posed to match atlas batch / library scale** in the generative-model sense—not raw counts.

**Earlier approach:** copy `layers["counts"]` into `.X`, then `normalize_total` + `log1p` to mirror notebook **15** (Scanpy on counts). **Current approach:** **do not overwrite `.X`**. We **subset and reorder** columns to atlas HVGs only, and **do not** apply `normalize_total` or `log1p` on BCG, on the grounds that **`.X` is already in the intended normalized style** relative to the atlas integration.

**Tradeoff (explicit):** Training data from notebook **15** uses **Scanpy** `normalize_total` + `log1p` on counts. Query data here uses **scANVI-derived `.X`**. That is a **deliberate domain choice**: prioritize alignment with the **mentor integration** over strict identity with the Scanpy-normalized training matrix. If evaluation is poor, compare against the count→`log1p` branch or retrain with a single consistent normalization.

**Note:** `layers["counts"]` (if present) is left on the loaded object for provenance only; **exports** use the projected **`.X`** values.

---

## Pipeline overview

1. Load BCG; keep **`.X`** as stored (`float32`); tag `obs`.
2. Load cached **one-to-one** ortholog table from notebook **15**; build **human ENSG → BCG column** using `bcg.var['gene_ids']` (ENSMUSG). No separate symbol→BioMart step.
3. For each flavor in `{seurat_v3, pearson_residuals}`: read `hvg_{flavor}_atlas_full_v07.h5ad` `var_names`, project BCG **`.X`** onto those genes (zeros for missing), **no further normalization**, write `bcg_mouse_aligned_{flavor}.h5ad`.
4. Round-trip via CellOT env → `*_v07.h5ad`.

**Dependency:** notebook **15** must be run first (HVG reference files + ortholog cache).

**Source:** `/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/bcg_mouse_aligned_050626.h5ad`


In [7]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp_sparse

sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")
from speciesot_helpers import strip_ensembl_gene_id

BASE_DIR = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT"
BCG_PATH = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/bcg_mouse_aligned_051026.h5ad"
DATASET_DIR = os.path.join(BASE_DIR, "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg")
ORTHO_CACHE = os.path.join(BASE_DIR, "scripts/.biomart_ortholog_cache.csv")
OUT_DIR = os.path.join(BASE_DIR, "speciesOT/baseline/analysis/bcg_mouse_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

FLAVORS = ["seurat_v3", "pearson_residuals"]
CELLOT_PY = "/n/home01/jzhou1125/.conda/envs/CellOT/bin/python"

print("BCG source:", BCG_PATH)
print("ortholog cache:", ORTHO_CACHE, "(exists?", os.path.exists(ORTHO_CACHE), ")")
print("dataset dir:", DATASET_DIR)
print("flavors:", FLAVORS)


/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BCG source: /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/bcg_mouse_aligned_051026.h5ad
ortholog cache: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/scripts/.biomart_ortholog_cache.csv (exists? True )
dataset dir: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg
flavors: ['seurat_v3', 'pearson_residuals']


## 1. Load BCG mouse (keep mentor `.X`)

Load the batch-mapped object from `tb1_mouse_setup` / Josh's export. We **do not** replace `.X` with `layers['counts']`.


In [8]:
bcg = sc.read_h5ad(BCG_PATH)
print("BCG read:", bcg.shape, "vars sample:", list(bcg.var_names[:5]))
print("obs cols:", list(bcg.obs.columns))
print("layers:", list(bcg.layers.keys()) if bcg.layers else "(none)")
assert "gene_ids" in bcg.var.columns, "expected BCG .var['gene_ids'] with ENSMUSG for ortholog join"
g0 = bcg.var["gene_ids"].astype(str).head(3)
print("  gene_ids sample:", dict(zip(bcg.var_names[:3], g0)))

# Keep mentor .X (scANVI-normalized / atlas-mapped expression). No log1p here.
if sp_sparse.issparse(bcg.X):
    bcg.X = bcg.X.astype(np.float32)
else:
    bcg.X = np.asarray(bcg.X, dtype=np.float32)

bcg.obs_names_make_unique()
bcg.obs["condition"] = "mouse"
bcg.obs["species"] = "mouse"

x_sample = bcg.X[:100].toarray().ravel() if sp_sparse.issparse(bcg.X) else bcg.X[:100].ravel()
print("\nUsing mentor .X (no count swap, no log1p in this notebook):")
print("  shape:", bcg.shape)
print(f"  .X dtype={bcg.X.dtype}, sample min={float(x_sample.min()):.4f}, max={float(x_sample.max()):.4f}, mean={float(x_sample.mean()):.4f}")
print(f"  cell_type counts: {bcg.obs['cell_type'].value_counts().to_dict()}")


BCG read: (1406, 10866) vars sample: ['Mrpl15', 'Lypla1', 'Tcea1', 'Atp6v1h', 'Rb1cc1']
obs cols: ['n_genes', 'leiden', 'cell_type', 'study', 'cell_type_original', 'cell_type_scanvi']
layers: ['counts']
  gene_ids sample: {'Mrpl15': 'ENSMUSG00000033845', 'Lypla1': 'ENSMUSG00000025903', 'Tcea1': 'ENSMUSG00000033813'}

Using mentor .X (no count swap, no log1p in this notebook):
  shape: (1406, 10866)
  .X dtype=float32, sample min=0.0000, max=9.1957, mean=0.2304
  cell_type counts: {'LT-HSC treated': 994, 'LT-HSC': 412}


/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [9]:
# Mentor .X = scANVI `get_normalized_expression`-style output (continuous, library-scaled / denoised).
# It is **not** Scanpy `log1p`, so do not expect a tight [0, ~10] band like typical log-normalized counts.
X_den = bcg.X.toarray() if sp_sparse.issparse(bcg.X) else np.asarray(bcg.X, dtype=np.float32)
flat = X_den.ravel()
qs = np.array([0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9, 100], dtype=float)
pct = np.percentile(flat, qs)
print("bcg.X full-matrix percentiles:")
for q, v in zip(qs, pct):
    print(f"  p{q:5.1f}: {v:.6g}")
print(f"  mean={flat.mean():.6g}  std={flat.std():.6g}  min={flat.min():.6g}  max={flat.max():.6g}")
print(f"  frac <= 0: {np.mean(flat <= 0):.4f}   n={flat.size:,} ({bcg.n_obs} cells × {bcg.n_vars} genes)")

bcg.X full-matrix percentiles:
  p  0.0: 0
  p  1.0: 0
  p  5.0: 0
  p 10.0: 0
  p 25.0: 0
  p 50.0: 0
  p 75.0: 0
  p 90.0: 0.940115
  p 95.0: 1.41567
  p 99.0: 3.04108
  p 99.9: 4.74334
  p100.0: 19.8487
  mean=0.250476  std=0.622994  min=0  max=19.8487
  frac <= 0: 0.7671   n=15,277,596 (1406 cells × 10866 genes)


### Outliers in mentor `.X` — **p99.9 is not the “second largest”**

**p99.9** = *99.9th percentile*: a cutoff such that **~99.9%** of the flattened entries are **≤** this value (depending slightly on the interpolation rule). It is **not** the second-largest entry; there can be **many** entries between p99.9 and the **maximum** (**p100**).

The next cell lists the **largest few** `(cell, gene, .X)` values, shows **`layers['counts']`** at those positions when the layer exists, then **replaces only the single global maximum** with the **p99.9** value to tame pathological spikes without rescaling the whole matrix. Uncomment or edit if you prefer a different rule (e.g. cap all values above p99.9).


In [10]:
# Dense working copy (sparse .X -> densified for clear row/col edits)
X_w = bcg.X.toarray().astype(np.float32, copy=False) if sp_sparse.issparse(bcg.X) else np.asarray(bcg.X, dtype=np.float32)
flat = X_w.ravel()
p999 = float(np.percentile(flat, 99.9))

k_top = 5
idx = np.argpartition(flat, -k_top)[-k_top:]
idx = idx[np.argsort(flat[idx])[::-1]]
print(f"Top-{k_top} entries in bcg.X:")
for rank, lin in enumerate(idx, 1):
    r, c = np.unravel_index(int(lin), X_w.shape)
    g, cl = bcg.var_names[c], bcg.obs_names[r]
    vx = float(X_w[r, c])
    extra = ""
    if "counts" in bcg.layers:
        C = bcg.layers["counts"]
        vc = float(C[r, c]) if not sp_sparse.issparse(C) else float(C[r, c])
        extra = f"  raw_counts={vc:.6g}"
    print(f"  #{rank}  cell={cl!r}  gene={g!r}  .X={vx:.6g}{extra}")

k_max = int(np.argmax(flat))
r_max, c_max = np.unravel_index(k_max, X_w.shape)
old_max = float(flat[k_max])
print(f"\nGlobal max (p100) = {old_max:.6g}  |  p99.9 = {p999:.6g}")

# Cap the single largest entry at p99.9 (one entry only; ties break by first argmax)
X_w[r_max, c_max] = p999
if sp_sparse.issparse(bcg.X):
    bcg.X = sp_sparse.csr_matrix(X_w)
else:
    bcg.X = X_w
print(
    f"Capped .X at ({bcg.obs_names[r_max]!r}, {bcg.var_names[c_max]!r}): "
    f"{old_max:.6g} -> {p999:.6g}"
)


Top-5 entries in bcg.X:
  #1  cell='TTTGGTTAGAGCTGGT-1'  gene='Plac8'  .X=19.8487  raw_counts=34
  #2  cell='AGTGAGGCAGGATTGG-1'  gene='Plac8'  .X=19.313  raw_counts=77
  #3  cell='GCGAGAATCGCACTCT-1'  gene='Plac8'  .X=18.9246  raw_counts=76
  #4  cell='GGCGTGTAGCTAAACA-1'  gene='Plac8'  .X=18.9079  raw_counts=65
  #5  cell='TTAGGACGTATAAACG-1'  gene='Plac8'  .X=18.8201  raw_counts=31

Global max (p100) = 19.8487  |  p99.9 = 4.74334
Capped .X at ('TTTGGTTAGAGCTGGT-1', 'Plac8'): 19.8487 -> 4.74334


### Atlas reference — same percentile summary on training `hvg_*_atlas_full_v07.h5ad`

Compare BCG mentor `.X` to the **actual** CellOT training matrices from notebook **15** (log-normalized counts in `.X`). Paths use `DATASET_DIR` + `FLAVORS` from the setup cell.


In [11]:
def _x_percentiles_report(adata, label):
    X = adata.X.toarray() if sp_sparse.issparse(adata.X) else np.asarray(adata.X, dtype=np.float32)
    flat = X.ravel()
    qs = np.array([0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9, 100], dtype=float)
    pct = np.percentile(flat, qs)
    print(f"{label}  shape={adata.shape}")
    for q, v in zip(qs, pct):
        print(f"  p{q:5.1f}: {v:.6g}")
    print(f"  mean={flat.mean():.6g}  std={flat.std():.6g}  min={flat.min():.6g}  max={flat.max():.6g}")
    print(f"  frac <= 0: {np.mean(flat <= 0):.4f}   n={flat.size:,}")


print("Atlas training HVG .X (notebook 15 → *_v07):")
for flavor in FLAVORS:
    atlas_path = os.path.join(DATASET_DIR, f"hvg_{flavor}_atlas_full_v07.h5ad")
    if not os.path.exists(atlas_path):
        print(f"\n  (missing) {atlas_path}\n")
        continue
    ad_at = sc.read_h5ad(atlas_path)
    print()
    _x_percentiles_report(ad_at, f"hvg_{flavor}_atlas_full_v07")


Atlas training HVG .X (notebook 15 → *_v07):

hvg_seurat_v3_atlas_full_v07  shape=(8610, 1000)
  p  0.0: 0
  p  1.0: 0
  p  5.0: 0
  p 10.0: 0
  p 25.0: 0
  p 50.0: 0
  p 75.0: 0
  p 90.0: 0
  p 95.0: 1.23692
  p 99.0: 2.72759
  p 99.9: 4.56242
  p100.0: 8.46476
  mean=0.1364  std=0.523838  min=0  max=8.46476
  frac <= 0: 0.9128   n=8,610,000

hvg_pearson_residuals_atlas_full_v07  shape=(8610, 1000)
  p  0.0: 0
  p  1.0: 0
  p  5.0: 0
  p 10.0: 0
  p 25.0: 0
  p 50.0: 0
  p 75.0: 0
  p 90.0: 2.11938
  p 95.0: 3.06811
  p 99.0: 4.29145
  p 99.9: 5.18595
  p100.0: 8.46476
  mean=0.477909  std=1.04083  min=0  max=8.46476
  frac <= 0: 0.7758   n=8,610,000


## 2. Cached ortholog table + `gene_ids` → human ENSG column index

Use **notebook 15**’s `human_ensembl_id` / `mouse_ensembl_id` pairs. Map each **human ENSG** to a BCG column via **`bcg.var['gene_ids']`** (strip version suffixes). **No** BioMart symbol lookup.


In [12]:
assert os.path.exists(ORTHO_CACHE), (
    f"BioMart ortholog cache not found at {ORTHO_CACHE}. "
    f"Please run 15_data_prep_full_atlas_no_holdout.ipynb first to generate it."
)
ortho_df = pd.read_csv(ORTHO_CACHE)
print(f"Loaded ortholog table: {len(ortho_df)} rows; columns: {list(ortho_df.columns)}")
if "orthology_type" in ortho_df.columns:
    ortho_df = ortho_df[ortho_df["orthology_type"] == "ortholog_one2one"].copy()
    print(f"  one2one only: {len(ortho_df)} rows")

ensg2musg = {}
for _, row in ortho_df.iterrows():
    h = strip_ensembl_gene_id(str(row["human_ensembl_id"]))
    m = strip_ensembl_gene_id(str(row["mouse_ensembl_id"]))
    if h and m:
        ensg2musg[h] = m
print(f"  human ENSG -> mouse ENSMUSG: {len(ensg2musg)} pairs")

musg2col = {}
for vi in range(bcg.n_vars):
    m = strip_ensembl_gene_id(str(bcg.var["gene_ids"].iloc[vi]))
    if not m:
        continue
    if m not in musg2col:
        musg2col[m] = vi
print(f"  ENSMUSG with a BCG column (first col wins): {len(musg2col)} / {bcg.n_vars} genes")

ensg_to_bcg_col = {}
for ensg, musg in ensg2musg.items():
    ci = musg2col.get(musg)
    if ci is not None:
        ensg_to_bcg_col[ensg] = ci

ex_hvg = os.path.join(DATASET_DIR, "hvg_seurat_v3_atlas_full_v07.h5ad")
if os.path.exists(ex_hvg):
    g1k = list(sc.read_h5ad(ex_hvg, backed="r").var_names.astype(str))
    g1k_s = [strip_ensembl_gene_id(str(g)) for g in g1k]
    n_hit = sum(1 for g in g1k_s if g in ensg_to_bcg_col)
    n_hvg_with_ortho = sum(1 for g in g1k_s if g in ensg2musg)
    n_mouse_missing = sum(
        1 for g in g1k_s if g in ensg2musg and ensg2musg[g] not in musg2col
    )
    print(f"  sanity: seurat atlas HVG genes with BCG column: {n_hit} / {len(g1k)}")
    print(
        f"  (HVGs with one2one ortholog: {n_hvg_with_ortho}; "
        f"of those, mouse ENSMUSG not in BCG gene_ids: {n_mouse_missing}; {n_hit} complete the mapping chain)"
    )


Loaded ortholog table: 14451 rows; columns: ['human_ensembl_id', 'human_gene_name', 'mouse_ensembl_id', 'mouse_gene_name', 'orthology_type']
  one2one only: 14451 rows
  human ENSG -> mouse ENSMUSG: 14451 pairs
  ENSMUSG with a BCG column (first col wins): 10866 / 10866 genes
  sanity: seurat atlas HVG genes with BCG column: 428 / 1000


## 3. Per-flavor HVG projection and export

For each atlas flavor, subset BCG **`.X`** to the same 1000 **ENSG** ids as `hvg_{flavor}_atlas_full_v07.h5ad` (zero-fill missing genes). **No** `normalize_total` or `log1p` — mentor `.X` is used as-is.


In [13]:
def make_bcg_for_flavor(bcg, ensg_to_bcg_col, atlas_hvg_path, out_path):
    """Project BCG `.X` onto the atlas's per-flavor HVG-1000 gene namespace (human ENSG order).
    Uses mentor `.X`; missing human ENSG -> zero column."""
    atlas = sc.read_h5ad(atlas_hvg_path)
    atlas_hvg_genes = list(atlas.var_names.astype(str))
    n_target = len(atlas_hvg_genes)
    print(f"  atlas HVG: {n_target} ENSG IDs")

    bcg_cols = []
    for ensg in atlas_hvg_genes:
        ci = ensg_to_bcg_col.get(ensg)
        bcg_cols.append(ci if ci is not None else -1)
    n_present = sum(1 for c in bcg_cols if c >= 0)
    print(f"  BCG coverage: {n_present} / {n_target} ({n_present/n_target:.1%}) of atlas HVG present in BCG")

    X_bcg = bcg.X
    if sp_sparse.issparse(X_bcg):
        X_bcg = X_bcg.toarray()
    X_new = np.zeros((bcg.n_obs, n_target), dtype=np.float32)
    for j, c in enumerate(bcg_cols):
        if c >= 0:
            X_new[:, j] = X_bcg[:, c]

    new_var = pd.DataFrame(index=pd.Index(atlas_hvg_genes, name="ensg"))
    new_obs = bcg.obs.copy()
    new_a = ad.AnnData(X=X_new, obs=new_obs, var=new_var)

    keep_cols = [c for c in ["condition", "species", "cell_type", "study", "_scvi_batch"] if c in new_a.obs.columns]
    new_a = ad.AnnData(X=new_a.X.astype(np.float32), obs=new_a.obs[keep_cols].copy(), var=new_a.var.copy())

    new_a.write_h5ad(out_path)
    print(f"  wrote {out_path}: shape {new_a.shape}, .X mean={float(new_a.X.mean()):.4f}, max={float(new_a.X.max()):.4f}")
    return n_present, n_target


coverage_rows = []
for flavor in FLAVORS:
    print(f"\n=== flavor: {flavor} ===")
    atlas_path = os.path.join(DATASET_DIR, f"hvg_{flavor}_atlas_full_v07.h5ad")
    out_path = os.path.join(DATASET_DIR, f"bcg_mouse_aligned_{flavor}.h5ad")
    n_pres, n_tot = make_bcg_for_flavor(bcg, ensg_to_bcg_col, atlas_path, out_path)
    coverage_rows.append({"flavor": flavor, "n_present": n_pres, "n_total": n_tot,
                          "coverage_pct": 100*n_pres/n_tot, "out_path": out_path})

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(os.path.join(OUT_DIR, "bcg_atlas_hvg_coverage.csv"), index=False)
print("\n=== Coverage summary ===")
print(coverage_df.to_string(index=False))



=== flavor: seurat_v3 ===
  atlas HVG: 1000 ENSG IDs
  BCG coverage: 428 / 1000 (42.8%) of atlas HVG present in BCG
  wrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_aligned_seurat_v3.h5ad: shape (1406, 1000), .X mean=0.0901, max=19.3130

=== flavor: pearson_residuals ===
  atlas HVG: 1000 ENSG IDs
  BCG coverage: 582 / 1000 (58.2%) of atlas HVG present in BCG
  wrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_aligned_pearson_residuals.h5ad: shape (1406, 1000), .X mean=0.3780, max=19.3130

=== Coverage summary ===
           flavor  n_present  n_total  coverage_pct                                                                                                                                     out_path
        seurat_v3        428     1000          42.8         /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-m

### BCG-missing HVG genes — expression in **atlas training** `.X`

For each flavor, genes that are **zero-filled** in BCG (no ENSMUSG in `gene_ids` / no column / no ortholog path) are listed with summary stats from **`hvg_{flavor}_atlas_full_v07.h5ad`**: how often nonzero in atlas, mean, median, max. **High `atlas_mean` or `atlas_frac_pos`** means hard zeros in BCG are a stronger distortion vs typical training. Full tables are written to **`bcg_mouse_outputs/`**.


In [14]:
def _missing_hvg_gene_indices(ensg_to_bcg_col, atlas_hvg_genes):
    """Indices j where atlas_hvg_genes[j] has no BCG column via ensg_to_bcg_col."""
    missing = []
    for j, ensg in enumerate(atlas_hvg_genes):
        if ensg not in ensg_to_bcg_col:
            missing.append(j)
    return missing


def report_missing_hvg_vs_atlas(ensg_to_bcg_col, atlas_hvg_path, flavor):
    ad_tr = sc.read_h5ad(atlas_hvg_path)
    genes = list(ad_tr.var_names.astype(str))
    miss_j = _missing_hvg_gene_indices(ensg_to_bcg_col, genes)
    X = ad_tr.X.toarray() if sp_sparse.issparse(ad_tr.X) else np.asarray(ad_tr.X, dtype=np.float32)
    sp = ad_tr.obs["species"].astype(str).values if "species" in ad_tr.obs.columns else None

    rows = []
    for j in miss_j:
        col = X[:, j]
        row = {
            "ensg": genes[j],
            "atlas_frac_pos": float(np.mean(col > 0)),
            "atlas_mean": float(np.mean(col)),
            "atlas_median": float(np.median(col)),
            "atlas_max": float(np.max(col)),
        }
        if sp is not None:
            row["atlas_mean_mouse"] = float(np.mean(col[sp == "mouse"]))
            row["atlas_mean_human"] = float(np.mean(col[sp == "human"]))
        rows.append(row)
    df = pd.DataFrame(rows).sort_values("atlas_mean", ascending=False)

    print(f"\n=== {flavor}: BCG-missing HVG → atlas training .X ({len(miss_j)} genes) ===")
    print("atlas file:", atlas_hvg_path)
    if df.empty:
        print("  (all 1000 HVG present in BCG — no zeros from missing genes)")
        return
    print("\nAggregate over missing genes (one row per gene):")
    print(f"  mean(atlas_mean):     {df['atlas_mean'].mean():.4f}")
    print(f"  median(atlas_mean):   {df['atlas_mean'].median():.4f}")
    print(f"  mean(atlas_frac_pos): {df['atlas_frac_pos'].mean():.4f}")
    print("\nTop 25 missing genes by atlas_mean (zero-fill is most misleading here):")
    print(df.head(25).to_string(index=False))
    print("\nSample: 15 lowest atlas_mean among missing (often near silent in atlas):")
    print(df.tail(15).to_string(index=False))
    out_csv = os.path.join(OUT_DIR, f"bcg_missing_hvg_atlas_expr_{flavor}.csv")
    df.to_csv(out_csv, index=False)
    print(f"\n  full table → {out_csv}")


for flavor in FLAVORS:
    atlas_p = os.path.join(DATASET_DIR, f"hvg_{flavor}_atlas_full_v07.h5ad")
    if os.path.exists(atlas_p):
        report_missing_hvg_vs_atlas(ensg_to_bcg_col, atlas_p, flavor)



=== seurat_v3: BCG-missing HVG → atlas training .X (572 genes) ===
atlas file: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_seurat_v3_atlas_full_v07.h5ad

Aggregate over missing genes (one row per gene):
  mean(atlas_mean):     0.0877
  median(atlas_mean):   0.0517
  mean(atlas_frac_pos): 0.0533

Top 25 missing genes by atlas_mean (zero-fill is most misleading here):
           ensg  atlas_frac_pos  atlas_mean  atlas_median  atlas_max  atlas_mean_mouse  atlas_mean_human
ENSG00000168484        0.301161    0.834180           0.0   8.458936          0.520440          1.147921
ENSG00000090382        0.244019    0.718766           0.0   6.401638          0.001715          1.435817
ENSG00000111341        0.243089    0.651765           0.0   7.534696          0.642531          0.660999
ENSG00000152583        0.215679    0.644694           0.0   5.991305          0.336547          0.952840
ENSG00000109321        0.271661    0.614847  

In [17]:
bcg

AnnData object with n_obs × n_vars = 1406 × 10866
    obs: 'n_genes', 'leiden', 'cell_type', 'study', 'cell_type_original', 'cell_type_scanvi', 'condition', 'species'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'gene_ids'
    uns: 'hvg', 'leiden', 'leiden_colors', 'neighbors', 'pca', 'study_colors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [22]:
bcg.var['gene_ids']

Mrpl15     ENSMUSG00000033845
Lypla1     ENSMUSG00000025903
Tcea1      ENSMUSG00000033813
Atp6v1h    ENSMUSG00000033793
Rb1cc1     ENSMUSG00000025907
                  ...        
Mid1       ENSMUSG00000035299
Kdm5d      ENSMUSG00000056673
Eif2s3y    ENSMUSG00000069049
Uty        ENSMUSG00000068457
Ddx3y      ENSMUSG00000069045
Name: gene_ids, Length: 10866, dtype: object

In [23]:
"Sftpc" in bcg.var['gene_ids']

False

## 4. Round-trip via CellOT env (anndata 0.7)

Copy → strip empty HDF5 groups → re-save with the CellOT interpreter so downstream tooling can load the files.


In [15]:
files = [os.path.join(DATASET_DIR, f"bcg_mouse_aligned_{f}.h5ad") for f in FLAVORS]
v07_paths = []
for src in files:
    dst = src.replace(".h5ad", "_v07.h5ad")
    if os.path.exists(dst):
        os.remove(dst)
    shutil.copy2(src, dst)
    v07_paths.append(dst)

strip_script = r"""
import sys, h5py
EMPTY = ["layers","obsm","obsp","uns","varm","varp"]
for p in sys.argv[1:]:
    with h5py.File(p, "r+") as f:
        for g in EMPTY:
            if g in f and len(f[g].keys())==0:
                del f[g]
        for a in ("encoding-type","encoding-version"):
            if a in f.attrs:
                del f.attrs[a]
"""

rewrite_script = r"""
import sys, os, h5py, numpy as np, pandas as pd, anndata as ad
from scipy import sparse
def _d(x): return x.decode() if isinstance(x,(bytes,np.bytes_)) else x
def load_obs(f):
    g = f["obs"]; idx = _d(g.attrs["_index"]) if "_index" in g.attrs else "index"
    index = [_d(x) for x in g[idx][:]]; cols={}
    for n in g.keys():
        if n==idx: continue
        node=g[n]
        if isinstance(node, h5py.Group) and "categories" in node and "codes" in node:
            cats=[_d(c) for c in node["categories"][:]]
            cols[n]=pd.Categorical.from_codes(node["codes"][:], categories=cats)
        else:
            arr=node[:]
            if arr.dtype.kind in ("O","S"):
                arr=np.array([_d(x) for x in arr])
            cols[n]=arr
    return pd.DataFrame(cols, index=pd.Index(index, name=idx))
def load_var(f):
    g = f["var"]; idx = _d(g.attrs["_index"]) if "_index" in g.attrs else "index"
    index = [_d(x) for x in g[idx][:]]
    cols={}
    for n in g.keys():
        if n==idx: continue
        node=g[n]
        arr = node[:]
        if arr.dtype.kind in ("O","S"):
            arr=np.array([_d(x) for x in arr])
        cols[n]=arr
    return pd.DataFrame(cols, index=pd.Index(index, name=idx))
def load_X(f):
    n=f["X"]
    if isinstance(n,h5py.Group):
        d=n["data"][:]; i=n["indices"][:]; p=n["indptr"][:]
        sh=tuple(n.attrs.get("shape", n.attrs.get("h5sparse_shape")))
        e=_d(n.attrs.get("encoding-type", b"csr_matrix"))
        return sparse.csc_matrix((d,i,p),shape=sh) if "csc" in e else sparse.csr_matrix((d,i,p),shape=sh)
    return n[:]
for p in sys.argv[1:]:
    with h5py.File(p,"r") as f:
        obs=load_obs(f); var=load_var(f); X=load_X(f)
    a=ad.AnnData(X=X,obs=obs,var=var)
    os.remove(p); a.write(p)
    print("rewrote",p,"shape",a.shape)
"""

subprocess.run([CELLOT_PY, "-c", strip_script, *v07_paths], check=True)
subprocess.run([CELLOT_PY, "-c", rewrite_script, *v07_paths], check=True)
print("\nBCG _v07 files now CellOT-env compatible:")
for p in v07_paths:
    print(f"  {p}")

rewrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_aligned_seurat_v3_v07.h5ad shape (1406, 1000)
rewrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_aligned_pearson_residuals_v07.h5ad shape (1406, 1000)

BCG _v07 files now CellOT-env compatible:
  /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_aligned_seurat_v3_v07.h5ad
  /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/bcg_mouse_aligned_pearson_residuals_v07.h5ad
